<a href="https://colab.research.google.com/github/Data-Creater-Atlas/Data-Atlas/blob/Seongjong/mission1__final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. 실행 환경 설정 및 모델 선택

from google.colab import drive
import os
from pathlib import Path

model_version = 'yolov9c'
batch_size = 8
epochs = 50

drive.mount('/content/drive')
!pip install ultralytics -q



Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.2 MB/s eta 0:00:00


In [3]:
# 2. 데이터셋 경로 설정

DATA_ROOT = Path("/content/drive/MyDrive/Data Camp/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터")

print("\n" + "="*40)
print(f"✅ 설정 완료! [{model_version}] 모델로 학습을 시작합니다.")
print(f" (배치 사이즈: {batch_size}, 에폭: {epochs})")



✅ 설정 완료! [yolov9c] 모델로 학습을 시작합니다.
 (배치 사이즈: 8, 에폭: 50)


In [4]:
# 3.  데이터셋 경로 지정 및 폴더 생성

# 원본 데이터
TS_DIR = DATA_ROOT / "Training" / "01.원천데이터" / "TS_KS"
TL_JSON_DIR = DATA_ROOT / "Training" / "02.라벨링데이터" / "TL_KS_BBOX"
VS_DIR = DATA_ROOT / "Validation" / "01.원천데이터" / "VS_KS"
VL_JSON_DIR = DATA_ROOT / "Validation" / "02.라벨링데이터" / "VL_KS_BBOX"

# YOLO 데이터셋이 저장될 위치
DATASET_DIR = DATA_ROOT / "YOLO_Dataset_Chimney" # 데이터셋 폴더명 지정
(TR_IMG_DIR := DATASET_DIR/"train/images").mkdir(parents=True, exist_ok=True)
(TR_LABEL_DIR := DATASET_DIR/"train/labels").mkdir(parents=True, exist_ok=True)
(VAL_IMG_DIR := DATASET_DIR/"valid/images").mkdir(parents=True, exist_ok=True)
(VAL_LABEL_DIR := DATASET_DIR/"valid/labels").mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["chimney"]

print("✅ YOLO 데이터셋 폴더 구조가 준비되었습니다.")

✅ YOLO 데이터셋 폴더 구조가 준비되었습니다.


In [5]:
# 4. 라벨링 데이터 변환 (VIA JSON → YOLO 포맷)
import json
import shutil
from tqdm.notebook import tqdm
from pathlib import Path

def _ensure_float(v):
    return float(str(v).strip())

def via_json_to_yolo(json_path: Path, images_dir: Path, out_img_dir: Path, out_lbl_dir: Path, class_id: int = 0):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for key, item in data.items():
        img_name = item.get("filename") or key
        if "." in img_name:
            stem, ext = img_name.rsplit(".", 1)
            img_name = stem + "." + ext

        src_img = images_dir / img_name
        if not src_img.exists():
            cands = list(images_dir.glob(Path(img_name).stem + ".*"))
            if cands: src_img = cands[0]
            else: continue

        fa = item.get("file_attributes", {})
        W = int(_ensure_float(fa.get("img_width", 0)))
        H = int(_ensure_float(fa.get("img_height", 0)))
        if W == 0 or H == 0: continue

        lines = []
        for region in item.get("regions", []):
            sa = region.get("shape_attributes", {})
            name = sa.get("name", "")
            if name == "rect":
                x, y = _ensure_float(sa["x"]), _ensure_float(sa["y"])
                bw, bh = _ensure_float(sa["width"]), _ensure_float(sa["height"])
            elif name == "polyline":
                xs, ys = sa.get("all_points_x", []), sa.get("all_points_y", [])
                if not xs or not ys: continue
                min_x, max_x = min(xs), max(xs)
                min_y, max_y = min(ys), max(ys)
                x, y = _ensure_float(min_x), _ensure_float(min_y)
                bw, bh = _ensure_float(max_x - min_x), _ensure_float(max_y - min_y)
            else:
                continue

            cx = max(0, min(1, (x + bw/2) / W))
            cy = max(0, min(1, (y + bh/2) / H))
            nw = max(0, min(1, bw / W))
            nh = max(0, min(1, bh / H))
            if nw <= 0 or nh <= 0: continue
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if lines:
            out_lbl = out_lbl_dir / f"{Path(img_name).stem}.txt"
            out_lbl.write_text("\n".join(lines), encoding="utf-8")
            dst_img = out_img_dir / src_img.name
            if not dst_img.exists():
                shutil.copy2(src_img, dst_img)

if len(list(TR_IMG_DIR.glob("*"))) == 0 or len(list(VAL_IMG_DIR.glob("*"))) == 0:
    print("라벨링 데이터 변환을 시작합니다. (시간이 다소 소요될 수 있습니다)")
    print("학습용 데이터 변환 중...")
    for jp in tqdm(sorted(TL_JSON_DIR.glob("*.json")), desc="Train JSON -> YOLO"):
        via_json_to_yolo(jp, TS_DIR, TR_IMG_DIR, TR_LABEL_DIR, class_id=0)
    print("검증용 데이터 변환 중...")
    for jp in tqdm(sorted(VL_JSON_DIR.glob("*.json")), desc="Validation JSON -> YOLO"):
        via_json_to_yolo(jp, VS_DIR, VAL_IMG_DIR, VAL_LABEL_DIR, class_id=0)
    print("\n✅ 데이터 변환 완료!")
else:
    print("✅ 이미 변환된 데이터가 있어 변환 과정을 건너뜁니다.")

✅ 이미 변환된 데이터가 있어 변환 과정을 건너뜁니다.


In [6]:
# 5. `data.yaml` 파일 생성
data_yaml = f"""
train: {str(TR_IMG_DIR)}
val: {str(VAL_IMG_DIR)}

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
""".strip()

(DATASET_DIR / "data.yaml").write_text(data_yaml, encoding="utf-8")
print("✅ data.yaml 파일이 성공적으로 생성되었습니다.")

✅ data.yaml 파일이 성공적으로 생성되었습니다.


In [7]:
# 6. YOLO 모델 학습 시작
from ultralytics import YOLO


model = YOLO(f"{model_version}.pt")

experiment_name = f"exp_{model_version}_e{epochs}_b{batch_size}_img512"

# 모델 학습
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=epochs,
    imgsz=512,
    batch=batch_size,
    patience=20,
    project=str(DATA_ROOT / "yolo_training_runs"),
    name=experiment_name,
    exist_ok=True,
    verbose=True,
    cache=True
)

#  경로 저장
last_experiment_path = results.save_dir
print(f"\n✅ 학습 완료! 결과 저장 경로: {last_experiment_path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Data Camp/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터/YOLO_Dataset_Chimney/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=

In [8]:
# 7. 최종 성능 평가 및 결과 출력
from ultralytics import YOLO
from pathlib import Path

best_model_path = Path(last_experiment_path) / "weights" / "best.pt"

model = YOLO(best_model_path)


val_results = model.val(
    data=str(DATASET_DIR/"data.yaml"),
    imgsz=512,
    split="val",
    verbose=False
)

# 성능 지표 추출 및 출력
precision = val_results.results_dict.get('metrics/precision(B)', 0)
recall = val_results.results_dict.get('metrics/recall(B)', 0)
map50 = val_results.results_dict.get('metrics/mAP50(B)', 0)
map50_95 = val_results.results_dict.get('metrics/mAP50-95(B)', 0)

print("\n" + "---" * 15)
print(f"🚀 최종 모델 성능 지표 ({model_version} | best.pt 기준)")
print("---" * 15)
print(f"  - Precision (정밀도)      : {precision:.4f}")
print(f"  - Recall (재현율)         : {recall:.4f}")
print(f"  - mAP@.5 (미션 평가 지표) : {map50:.4f}")
print(f"  - mAP@.5:.95             : {map50_95:.4f}")
print("---" * 15)

Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
YOLOv9c summary (fused): 156 layers, 25,320,019 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 60.8±16.4 MB/s, size: 104.6 KB)
val: Scanning /content/drive/MyDrive/Data Camp/53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터/YOLO_Dataset_Chimney/valid/labels.cache... 1006 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1006/1006 994.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 5.7it/s 11.1s
                   all       1006       1322      0.975       0.98      0.993      0.852
Speed: 0.2ms preprocess, 2.9ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val

---------------------------------------------
🚀 최종 모델 성능 지표 (yolov9c | best.pt 기준)
---------------------------------------------
  - Precision (정밀도)      : 0.9746
  - Recall (재현율)      

In [9]:
!pip freeze | grep -E "ultralytics|torch|torchvision|matplotlib|pandas|numpy|tqdm" > requirements.txt